In [3]:
import sys
print(sys.executable)

c:\Users\naiks\Databrickstolangchain\.venv\Scripts\python.exe


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import databricks
print(dir(databricks))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__']


In [ ]:
from databricks import sqlalchemy as databricks_sqlalchemy
print("ok")

In [ ]:
from databricks import sqlalchemy as databricks_sqlalchemy
print("ok")

In [17]:
from langchain_community.utilities import SQLDatabase
token = os.getenv("DATABRICKS_TOKEN")
host = os.getenv("DATABRICKS_HOST")
http_path = os.getenv("DATABRICKS_HTTP_PATH")
catalog = os.getenv("DATABRICKS_CATALOG")
schema = os.getenv("DATABRICKS_SCHEMA")

db = SQLDatabase.from_uri(
    f"databricks://token:{token}@{host}?"
    f"http_path={http_path}&catalog={catalog}&schema={schema}"
)

[WARN] Parameter '_user_agent_entry' is deprecated; use 'user_agent_entry' instead. This parameter will be removed in the upcoming releases.


In [20]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

toolkit = SQLDatabaseToolkit(db=db, llm=llm)
sql_tools = toolkit.get_tools()

In [21]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=sql_tools,
    system_prompt="""
    You are a data assistant. Use SQL tools to answer questions
    from Databricks Gold tables. Generate safe SQL queries and explain results.
    """
)

In [24]:
# Sales analysis
response = agent.invoke({
    "input": """Query the sales_by_policytype_month table and return the total premium 
    collected for each policy type summed across all months. 
    Present the results in a clean formatted table with policy type and total premium columns only."""
})
print(response["messages"][-1].content)


Here are the schemas and sample rows for the three tables in the database:

### 1. `claim_analysis`
- **Schema:**
  - `policy_type` STRING
  - `claim_status` STRING
  - `avg_claim_amount` INT
  - `max_claim_amount` INT
  - `min_claim_amount` INT
  - `total_claims` INT
  - `updated_timestamp` TIMESTAMP

- **Sample Rows:**
  | policy_type | claim_status | avg_claim_amount | max_claim_amount | min_claim_amount | total_claims | updated_timestamp               |
  |-------------|--------------|-------------------|-------------------|-------------------|---------------|----------------------------------|
  | Auto        | None         | 99835             | 197403            | 2283              | 351           | 2026-05-05 00:52:44.336032+00:00 |
  | Health      | None         | 97129             | 199995            | 2188              | 350           | 2026-05-05 00:52:44.336032+00:00 |

### 2. `sales_by_policytype_month`
- **Schema:**
  - `policy_type` STRING
  - `sale_month` STRING
  - `to

Here are the schemas for the three tables in the database:

### 1. `claim_analysis`
- **Columns:**
  - `policy_type` (STRING): Type of insurance policy.
  - `claim_status` (STRING): Status of the claim (e.g., approved, denied).
  - `avg_claim_amount` (INT): Average amount of claims for the policy type.
  - `max_claim_amount` (INT): Maximum claim amount for the policy type.
  - `min_claim_amount` (INT): Minimum claim amount for the policy type.
  - `total_claims` (INT): Total number of claims for the policy type.
  - `updated_timestamp` (TIMESTAMP): Last updated timestamp for the record.

### 2. `sales_by_policytype_month`
- **Columns:**
  - `policy_type` (STRING): Type of insurance policy.
  - `sale_month` (STRING): Month of the sales.
  - `total_premium` (INT): Total premium collected for the policy type in that month.
  - `updated_timestamp` (TIMESTAMP): Last updated timestamp for the record.

### 3. `sales_by_policytype_status`
- **Columns:**
  - `policy_type` (STRING): Type of insu